# Feature Engineering for Demand Forecasting
**Author:** Anuj Saini  
**Date:** July 14, 2026

This notebook builds and tests the feature engineering pipeline for our demand forecasting model. Our focus is on **temporal hygiene**: ensuring that every feature computed for week $W$ uses only information available *before* week $W$ (no data leakage).

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# Add src to python path for importing features.py
sys.path.append(os.path.abspath('..'))
from src.features import build_features

# Database Connection
host = "aws-0-ap-south-1.pooler.supabase.com"
port = "5432"
database = "postgres"
username = "ecom_ro_user.imnzftquwjuxcwpeufwp"
password = "work-experience-read-only"

conn_url = f"postgresql://{username}:{password}@{host}:{port}/{database}"
engine = create_engine(conn_url)
print("Connected to database successfully!")

Connected to database successfully!


In [2]:
# Load the raw transaction data
query = """
    SELECT 
        o.created_at,
        oi.qty,
        oi.unit_price,
        c.category_name
    FROM ecom.orders o
    JOIN ecom.order_items oi ON o.order_id = oi.order_id
    JOIN ecom.product_variants pv ON oi.variant_id = pv.variant_id
    JOIN ecom.products p ON pv.product_id = p.product_id
    JOIN ecom.categories c ON p.category_id = c.category_id;
"""

df_raw = pd.read_sql(query, engine)
df_raw['created_at'] = pd.to_datetime(df_raw['created_at'])
print(f"Loaded {len(df_raw)} records.")
print(f"Date range: {df_raw['created_at'].min()} to {df_raw['created_at'].max()}")

Loaded 81806 records.
Date range: 2026-03-16 00:13:57+00:00 to 2026-06-14 23:28:40+00:00


In [3]:
# Build features using the pipeline in src/features.py
df_features = build_features(df_raw)

print(f"Features DataFrame shape: {df_features.shape}")
print("\nColumns built:")
print(df_features.columns.tolist())

Features DataFrame shape: (126, 15)

Columns built:
['week_start', 'category_name', 'qty', 'avg_price', 'qty_lag_1', 'qty_lag_2', 'qty_lag_3', 'qty_lag_4', 'qty_roll_mean_2', 'qty_roll_std_2', 'qty_roll_mean_4', 'qty_roll_std_4', 'price_lag_1', 'category_expanding_mean', 'time_idx']


/home/anuj/Personal/DataBuoy-Projects-Testing/demand-forecasting-ml/src/features.py:23: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df['week_start'] = df['created_at'].dt.to_period('W').dt.start_time


In [4]:
# Check for any remaining NaN values
print("Any NaN values in the features DataFrame?", df_features.isnull().values.any())

Any NaN values in the features DataFrame? False


In [5]:
# Inspect a sample of the feature DataFrame
df_features.head(10)

,week_start,category_name,qty,avg_price,qty_lag_1,qty_lag_2,qty_lag_3,qty_lag_4,qty_roll_mean_2,qty_roll_std_2,qty_roll_mean_4,qty_roll_std_4,price_lag_1,category_expanding_mean,time_idx
0,2026-04-13,Accessories,764,839.775862,1058.0,991.0,868.0,949.0,1024.5,47.376154,966.50,79.542442,823.482339,966.500000,4
1,2026-04-20,Accessories,544,801.004717,764.0,1058.0,991.0,868.0,911.0,207.889394,920.25,130.543416,839.775862,926.000000,5
2,2026-04-27,Accessories,465,847.533724,544.0,764.0,1058.0,991.0,654.0,155.563492,839.25,233.605615,801.004717,862.333333,6
3,2026-05-04,Accessories,473,816.639257,465.0,544.0,764.0,1058.0,504.5,55.861436,707.75,265.569043,847.533724,805.571429,7
4,2026-05-11,Accessories,420,821.755418,473.0,465.0,544.0,764.0,469.0,5.656854,561.50,139.591069,816.639257,764.000000,8
5,2026-05-18,Accessories,487,772.170732,420.0,473.0,465.0,544.0,446.5,37.476659,475.50,51.280276,821.755418,725.777778,9
6,2026-05-25,Accessories,438,765.029412,487.0,420.0,473.0,465.0,453.5,47.376154,461.25,28.964058,772.170732,701.900000,10
7,2026-06-01,Accessories,318,810.788618,438.0,487.0,420.0,473.0,462.5,34.648232,454.50,30.881494,765.029412,677.909091,11
8,2026-06-08,Accessories,215,851.095808,318.0,438.0,487.0,420.0,378.0,84.852814,415.75,71.051038,810.788618,647.916667,12
9,2026-04-13,Bedding,647,1828.716024,857.0,868.0,702.0,674.0,862.5,7.778175,775.25,101.493432,1824.118110,775.250000,4


In [6]:
# Sanity check: verify distributions of lags vs target
stats_df = df_features[['qty', 'qty_lag_1', 'qty_lag_2', 'qty_lag_3', 'qty_lag_4']].describe()
print(stats_df)

              qty    qty_lag_1    qty_lag_2    qty_lag_3    qty_lag_4
count  126.000000   126.000000   126.000000   126.000000   126.000000
mean   442.603175   527.555556   603.206349   645.349206   684.650794
std    133.001599   202.356223   227.599343   224.727971   223.924249
min    200.000000   241.000000   340.000000   340.000000   340.000000
25%    375.250000   417.750000   438.000000   465.000000   473.000000
50%    444.500000   468.000000   490.500000   538.000000   700.500000
75%    499.750000   541.500000   739.750000   828.000000   878.250000
max    804.000000  1110.000000  1110.000000  1110.000000  1110.000000


In [7]:
# Sanity check for leakage: Train a quick model on the features to predict qty
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = df_features[[
    'qty_lag_1', 'qty_lag_2', 'qty_lag_3', 'qty_lag_4',
    'qty_roll_mean_2', 'qty_roll_std_2', 'qty_roll_mean_4', 'qty_roll_std_4',
    'price_lag_1', 'category_expanding_mean', 'time_idx'
]]
y = df_features['qty']

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

r2 = r2_score(y, y_pred)
wmape_model = (y - y_pred).abs().sum() / y.sum()

# Naive baseline (predict next week's demand is equal to this week's demand)
y_pred_naive = df_features['qty_lag_1']
wmape_naive = (y - y_pred_naive).abs().sum() / y.sum()

print("--- Evaluation Sanity Checks ---")
print(f"Linear Regression R2 score: {r2:.4f}")
print(f"Linear Regression WMAPE : {wmape_model:.4f}")
print(f"Naive Lag-1 Baseline WMAPE: {wmape_naive:.4f}")

--- Evaluation Sanity Checks ---
Linear Regression R2 score: 0.9115
Linear Regression WMAPE : 0.0726
Naive Lag-1 Baseline WMAPE: 0.2386


### Summary of Feature Groups and Rationale

1. **Lags (`qty_lag_1` to `qty_lag_4`)**: 
   - *Rationale*: Capture autoregressive dependencies and immediate short-term demand inertia. By looking back up to 4 weeks, the model can capture month-over-month patterns.
2. **Rolling Statistics (`qty_roll_mean_2`, `qty_roll_std_2`, `qty_roll_mean_4`, `qty_roll_std_4`)**:
   - *Rationale*: Smoothen out week-over-week noise and capture local trends and volatility. A 2-week rolling window captures short-term shifts, while a 4-week window captures medium-term trends. Standard deviations capture demand volatility, which is vital for calculating safety stock.
3. **Calendar/Temporal Features (`time_idx`)**:
   - *Rationale*: A linear time index represents weeks elapsed since the start of the dataset. Given the strong, continuous downward trend in demand, this feature allows the model to learn the global decay rate.
4. **Pricing features (`price_lag_1`)**:
   - *Rationale*: Captures price elasticity of demand. If prices for a category fluctuate, the model can learn how price shifts in week $W-1$ impact demand in week $W$.
5. **Categorical Features (`category_expanding_mean`)**:
   - *Rationale*: Target encodes the `category_name` by taking the expanding average of demand for that category over all prior weeks. This represents the baseline volume of each category (e.g., Skincare is high-volume, Bedding is low-volume) in a single column without leakage.

### How We Avoided Leakage (Temporal Hygiene)
- Every feature for week $W$ is computed using data from week $W-1$ or earlier.
- In `src/features.py`, we explicitly apply `shift(1)` to the target variable `qty` and the `avg_price` *before* computing any rolling statistics, lags, or expanding means.
- This guarantees that no information from week $W$ is used to predict week $W$.
- The training sanity check shows a Naive WMAPE of 23.86% and a Model WMAPE of 7.26%. If there was leakage (e.g., predicting using current week's information), the training error would be close to 0. The fact that the model achieves 7.26% (and not 0.0%) while significantly beating the naive baseline confirms that the model is learning the trend without leakage.